In [ ]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
if "snakemake" not in globals():
        from _helpers_notebooks import mock_snakemake
        snakemake = mock_snakemake(
                "plot_magnitude_pull",
                scenario="default", # default, penalty-sa, penalty-ea, penalty-nwa, penalty-oc
                wacc="regional", # regional, uniform
                cost_year="2050",
                interone="hbi",
                trade_chain="supplyconstraint"
        )

In [ ]:
add_iron_ore_cost = True
ironorecost = 97 * 1.59

In [ ]:
config = snakemake.config

### Get steel demand

In [ ]:
steel_demand = pd.read_csv(snakemake.input.steel_demand, index_col="region")

### Get costs

In [ ]:
df_all = pd.DataFrame()

# Select input files directly from snakemake (avoids hardcoded paths)
input_files = snakemake.input.supply_curves_interone

for fn in input_files:
    # Extract region from filename pattern: {region}_{process}.csv
    region = fn.split("/")[-1].split("_marginal_cost")[0]

    df = pd.read_csv(fn)
    df["region"] = region

    # Calculate the difference in demand to get the quantity supplied at each cost step
    diff = df["demand [t]"].diff()
    diff[0] = diff[1]  # set first value to second to avoid NaN
    df["demand [t]"] = diff

    df_all = pd.concat([df_all, df], ignore_index=True)

# Remove rows where the lcox [EUR/t] value is NaN (e.g. due to missing data for some regions)
df_all = df_all.dropna(subset=["lcox [EUR/t]"])


if add_iron_ore_cost:
    df_all["lcox [EUR/t]"] += ironorecost

df_all.set_index("region", inplace=True)

In [ ]:
costs = df_all[["lcox [EUR/t]", "product_demand_mt"]]

#### Get costs at demand level

In [ ]:
def interpolate_cost(region, demand, costs):
    if region not in costs.index:
        return np.nan

    sub = costs.loc[[region]].sort_values("product_demand_mt")
    demands = sub["product_demand_mt"].values
    lcox = sub["lcox [EUR/t]"].values

    # exact match
    if demand in demands:
        return lcox[demands == demand].mean()

    # below smallest or above largest -> take the closest edge value
    if demand <= demands[0]:
        return lcox[0]
    if demand >= demands[-1]:
        return lcox[-1]

    # find bracketing points
    idx_upper = np.searchsorted(demands, demand)
    idx_lower = idx_upper - 1

    return (lcox[idx_lower] + lcox[idx_upper]) / 2

In [ ]:
steel_demand["lcox_EUR_per_t"] = steel_demand.apply(
    lambda row: interpolate_cost(row.name, row["SteelDemand_DRI_Mt"], costs),
    axis=1
)
steel_cost_demand = steel_demand

### Steel production capacities

In [ ]:
steel_production = pd.read_csv(snakemake.input.steel_production)

In [ ]:
# Build ISO3 -> region mapping from config
regions = config["regions"]
iso3_to_region = {code: region for region, codes in regions.items() for code in codes}

# Map each country row to its region
steel_production_new = steel_production.copy()
steel_production_new["region"] = steel_production_new["iso3"].map(
    lambda x: iso3_to_region.get(x, "Other")
)

# Sum crude steel production and population per region, then compute per-capita
steel_production_by_region = (
    steel_production_new
    .groupby("region")[["crude_steel_production_t", "population"]]
    .sum()
)
steel_production_by_region["SteelProductionMt"] = (
    steel_production_by_region["crude_steel_production_t"] / 1e6
)
steel_production_by_region["SteelProductionMtPerCapita"] = (
    steel_production_by_region["crude_steel_production_t"]
    / steel_production_by_region["population"]
)
steel_production_by_region = steel_production_by_region.sort_values(
    "SteelProductionMt", ascending=False
)

steel_production_by_region[["SteelProductionMt", "SteelProductionMtPerCapita"]]


## Plot

In [ ]:
import seaborn as sns
from matplotlib.lines import Line2D

# --- user-facing plotting controls ---
plot_style = {
    "neutral_color": "#797777",
    "show_axes_box": True,  # toggle outer box/frame
}

# Base label position: always above bubble. Per-region x/y are additional offsets.
base_label_offset = {"x": 0, "y": 7}

# --- merge production capacity + cost/demand on region ---
plot_df = steel_cost_demand.join(
    steel_production_by_region[["SteelProductionMtPerCapita"]], how="inner"
)
plot_df = plot_df.reset_index()  # region: index -> column

nice_names = config.get("region_nice_names", {})
plot_df["label"] = plot_df["region"].map(lambda r: nice_names.get(r, r))

# --- per-region manual label controls (hard-coded block) ---
# x, y: additional offset points relative to base top position; line: draw connector line
label_layout = {
    "Central_America": {"x": 30, "y": 3, "line": False},
    "East_Asia": {"x": -24, "y": 10, "line": False},
    "East_East_Asia": {"x": 0, "y": 0, "line": False},
    "Eurasia": {"x": 20, "y": 0, "line": False},
    "Europe": {"x": 0, "y": 5, "line": False},
    "Far_West_Europe": {"x": 30, "y": -40, "line": True},
    "Middle_East": {"x": 0, "y": 0, "line": False},
    "North_America": {"x": 0, "y": 0, "line": False},
    "North_West_Africa": {"x": 16, "y": -30, "line": True},
    "Oceania": {"x": 0, "y": 0, "line": False},
    "Other": {"x": 0, "y": 0, "line": False},
    "Pacific_Asia": {"x": -20, "y": 20, "line": True},
    "South_America": {"x": 0, "y": 5, "line": False},
    "South_South_America": {"x": -10, "y": -30, "line": True},
    "Subsaharan_Africa": {"x": 15, "y": 5, "line": False},
    "West_Asia": {"x": 35, "y": 15, "line": False},
}

# --- quadrant thresholds (medians) ---
x_mid = plot_df["SteelProductionMtPerCapita"].median()
y_mid = plot_df["lcox_EUR_per_t"].median()

# --- bubble sizes: area proportional to DRI demand ---
max_bubble = 2500
demand_max = plot_df["SteelDemand_DRI_Mt"].max()
plot_df["bubble"] = (plot_df["SteelDemand_DRI_Mt"] / demand_max * max_bubble).clip(lower=30)

# --- colour palette (one colour per region) ---
palette = sns.color_palette("tab20", n_colors=len(plot_df))

fig, ax = plt.subplots(figsize=(8.8, 6.2))

for i, (_, row) in enumerate(plot_df.iterrows()):
    x_val = row["SteelProductionMtPerCapita"]
    y_val = row["lcox_EUR_per_t"]
    bubble_area = row["bubble"]
    label_text = row["label"]
    region_key = row["region"]
    point_color = palette[i]

    ax.scatter(
        x_val,
        y_val,
        s=bubble_area,
        color=point_color,
        alpha=0.8,
        edgecolors="white",
        linewidths=0.8,
        zorder=3,
    )

    region_cfg = label_layout.get(region_key, {"x": 0, "y": 0, "line": False})
    label_offset = (
        base_label_offset["x"] + region_cfg.get("x", 0),
        base_label_offset["y"] + region_cfg.get("y", 0),
    )
    draw_line = bool(region_cfg.get("line", False))

    annotate_kwargs = {
        "xy": (x_val, y_val),
        "xytext": label_offset,
        "textcoords": "offset points",
        "fontsize": 8.5,
        "color": point_color,
        "ha": "center",
        "va": "bottom",
        "zorder": 4,
    }
    if draw_line:
        annotate_kwargs["arrowprops"] = {
            "arrowstyle": "-",
            "lw": 0.8,
            "color": plot_style["neutral_color"],
            "alpha": 0.9,
            "shrinkA": 2,
            "shrinkB": 7,
        }

    ax.annotate(label_text, **annotate_kwargs)

# --- quadrant dividers ---
# ax.axvline(x_mid, color=plot_style["neutral_color"], linestyle="--", linewidth=1.0, alpha=0.6, zorder=2)
# ax.axhline(y_mid, color=plot_style["neutral_color"], linestyle="--", linewidth=1.0, alpha=0.6, zorder=2)

ax.set_ylim(600, 850)
ax.set_xlim(-0.09, 0.8)

# --- quadrant text annotations ---
xl, xr = ax.get_xlim()
yb, yt = ax.get_ylim()
px = (xr - xl) * 0.02
py = (yt - yb) * 0.02

quadrant_labels = [
    (xl + px, yt - py, "top", "left", "economically disadvantaged but\nlimited transition exposure"),
    (xr - px, yt - py, "top", "right", "most exposed regions"),
    (xl + px, yb + py, "bottom", "left", "potential emerging\ngreen-iron exporters"),
    (xr - px, yb + py, "bottom", "right", "comparatively well\npositioned to transition"),
]
for xq, yq, va, ha, txt in quadrant_labels:
    ax.text(
        xq,
        yq,
        txt,
        fontsize=8,
        va=va,
        ha=ha,
        color=plot_style["neutral_color"],
        style="italic",
        alpha=0.9,
        zorder=2,
    )

# --- axis labels (no title) ---
ax.set_xlabel(
    "Today's Steel Production Capacity (t$_{hbi}$/per capita)",
    fontsize=12,
    color=plot_style["neutral_color"],
)
ax.set_ylabel(
    "Domestic HBI costs in 2050 in €/t$_{hbi}$ \nas a proxy for the magnitude of Renewables Pull",
    fontsize=12,
    color=plot_style["neutral_color"],
)

# --- use neutral colour for all non-bubble elements ---
ax.tick_params(axis="both", colors=plot_style["neutral_color"])
for spine in ax.spines.values():
    spine.set_visible(plot_style["show_axes_box"])
    spine.set_color(plot_style["neutral_color"])

# --- bubble-size legend outside right; spaced markers; no frame ---
legend_values = [10, 50, 100]
legend_handles = []
for mt_val in legend_values:
    s = (mt_val / demand_max) * max_bubble
    marker_size = np.sqrt(s) * 0.72  # convert area-like scale to marker size
    legend_handles.append(
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markerfacecolor=plot_style["neutral_color"],
            markeredgecolor=plot_style["neutral_color"],
            alpha=0.55,
            markersize=marker_size,
            label=f"{mt_val} Mt",
        )
    )

legend = ax.legend(
    handles=legend_handles,
    title="HBI Demand in 2050",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=9,
    title_fontsize=9,
    labelspacing=1.6,
    handletextpad=1.2,
    borderaxespad=1.2,
)
legend.get_title().set_color(plot_style["neutral_color"])
for t in legend.get_texts():
    t.set_color(plot_style["neutral_color"])

plt.tight_layout()
plt.savefig(snakemake.output.magnitude_pull_png, dpi=150, bbox_inches="tight")
plt.savefig(snakemake.output.magnitude_pull, bbox_inches="tight")
plt.show()